# KL-Constrained Baum-Welch Under HMM Order Misspecification

**Goal**: Compare standard Baum-Welch vs KL-constrained BW for fitting first-order HMMs to higher-order Markov data.

**Key Results**: KL constraints improve robustness under misspecification while maintaining parameter recovery.

In [ ]:
import subprocess, sys
pkgs = ['numpy', 'scipy', 'matplotlib', 'loguru', 'pandas']
try:
    import google.colab
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('✓ Dependencies installed')

In [ ]:
import json, sys, math, time
from pathlib import Path
from typing import Optional
import numpy as np
from scipy import stats as sp_stats
from scipy.optimize import curve_fit
from loguru import logger
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
logger.remove()

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-797861-robust-parameter-learning-under-hmm-orde/main/iter_0/gen_art_experiment_1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except:
        pass
    import os
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('Could not load data')

In [ ]:
data = load_data()
print(f'✓ Loaded {len(data["datasets"][0]["examples"])} examples')
print(f'✓ Metadata: {data["metadata"]["method_name"]}')

In [ ]:
# Minimal configuration for demo
D = 4
OBS_DIM = 3
K_TRUES = [2]
T_VALUES = [100]
SEEDS = [0, 1]
N_TRAIN = 5
N_TEST = 3
EPSILONS = [0.01, 0.1]
N_RESTARTS = 2
MAX_ITER = 10
CONV_TOL = 1e-5
BIN_SEARCH_TOL = 1e-4
BIN_SEARCH_MAXITER = 60
LOG2PI = math.log(2 * math.pi)
EPS_FLOAT = 1e-300

print(f'Config: K_TRUES={K_TRUES}, T_VALUES={T_VALUES}, N_TRAIN={N_TRAIN}, MAX_ITER={MAX_ITER}')

## Data Generation
Generate synthetic k-th order Markov sequences

In [ ]:
def make_random_hmm_params(rng, d, obs_dim):
    A = rng.dirichlet(np.ones(d) * 2, size=d)
    pi = rng.dirichlet(np.ones(d) * 2)
    mu = rng.standard_normal((d, obs_dim)) * 2.5
    sigma = np.exp(rng.standard_normal((d, obs_dim)) * 0.3) * 0.8
    return {'A': A, 'pi': pi, 'mu': mu, 'sigma': sigma}

def make_kth_order_transition(rng, d, k):
    n_contexts = d ** k
    rows = rng.dirichlet(np.ones(d) * 1.5, size=n_contexts)
    return rows.reshape((d,) * k + (d,))

def generate_sequences(rng, n_seqs, T, k_true, params, trans_tensor):
    d = params['A'].shape[0]
    obs_dim = params['mu'].shape[1]
    mu, sigma, pi, A = params['mu'], params['sigma'], params['pi'], params['A']
    obs_list, states_list = [], []
    for _ in range(n_seqs):
        states = np.empty(T, dtype=int)
        obs = np.empty((T, obs_dim))
        for t in range(min(k_true, T)):
            states[t] = (rng.choice(d, p=pi) if t == 0 else rng.choice(d, p=A[states[t-1]]))
            obs[t] = mu[states[t]] + sigma[states[t]] * rng.standard_normal(obs_dim)
        for t in range(k_true, T):
            if k_true == 1: probs = trans_tensor[states[t-1]]
            elif k_true == 2: probs = trans_tensor[states[t-1], states[t-2]]
            else: probs = trans_tensor[states[t-1], states[t-2], states[t-3]]
            probs = np.maximum(probs, 1e-10); probs /= probs.sum()
            states[t] = rng.choice(d, p=probs)
            obs[t] = mu[states[t]] + sigma[states[t]] * rng.standard_normal(obs_dim)
        obs_list.append(obs); states_list.append(states)
    return obs_list, states_list

print('✓ Data generation functions defined')

## Forward-Backward Algorithm
Batched scaled E-step (Rabiner 1989)

In [ ]:
def emission_prob(obs_arr, mu, sigma):
    single = obs_arr.ndim == 2
    if single: obs_arr = obs_arr[None]
    N, T, obs_dim = obs_arr.shape
    d = mu.shape[0]
    diff = obs_arr[:, :, None, :] - mu[None, None, :, :]
    log_sig = np.log(np.maximum(sigma, 1e-8))
    log_const = -0.5 * obs_dim * LOG2PI - log_sig.sum(1)
    mahal = 0.5 * (diff / np.maximum(sigma[None, None], 1e-8)) ** 2
    log_B = log_const[None, None, :] - mahal.sum(-1)
    B = np.exp(log_B - log_B.max(-1, keepdims=True))
    B = np.maximum(B, EPS_FLOAT)
    return B[0] if single else B

def batched_e_step(obs_array, A, pi, mu, sigma):
    N, T, obs_dim = obs_array.shape
    d = A.shape[0]
    B = emission_prob(obs_array, mu, sigma)
    alpha, scales = np.empty((N, T, d)), np.empty((N, T))
    alpha[:, 0] = pi[None] * B[:, 0]
    scales[:, 0] = alpha[:, 0].sum(1)
    alpha[:, 0] /= np.maximum(scales[:, 0, None], EPS_FLOAT)
    for t in range(1, T):
        alpha[:, t] = B[:, t] * (alpha[:, t-1] @ A)
        scales[:, t] = alpha[:, t].sum(1)
        alpha[:, t] /= np.maximum(scales[:, t, None], EPS_FLOAT)
    total_log_ll = float(np.log(np.maximum(scales, EPS_FLOAT)).sum())
    beta = np.ones((N, T, d))
    for t in range(T-2, -1, -1):
        beta[:, t] = ((B[:, t+1] * beta[:, t+1]) @ A.T / np.maximum(scales[:, t+1, None], EPS_FLOAT))
    gamma = alpha * beta
    gamma /= np.maximum(gamma.sum(-1, keepdims=True), EPS_FLOAT)
    Y = (B[:, 1:] * beta[:, 1:] / np.maximum(scales[:, 1:, None], EPS_FLOAT))
    XY = alpha[:, :-1].reshape(-1, d).T @ Y.reshape(-1, d)
    C_A_num = A * XY
    C_pi = gamma[:, 0].sum(0)
    C_w = gamma.sum(axis=(0, 1))
    C_wx = np.einsum('ntj,ntk->jk', gamma, obs_array)
    C_wx2 = np.einsum('ntj,ntk->jk', gamma, obs_array ** 2)
    return C_A_num, C_pi, C_w, C_wx, C_wx2, total_log_ll

print('✓ Forward-backward function defined')

## M-Steps
Standard and KL-constrained updates

In [ ]:
def m_step_standard(C_A_num, C_pi, C_w, C_wx, C_wx2):
    A = np.maximum(C_A_num / np.maximum(C_A_num.sum(1, keepdims=True), EPS_FLOAT), EPS_FLOAT)
    A /= A.sum(1, keepdims=True)
    pi = np.maximum(C_pi / max(C_pi.sum(), EPS_FLOAT), EPS_FLOAT); pi /= pi.sum()
    w = np.maximum(C_w[:, None], EPS_FLOAT)
    mu = C_wx / w
    var = np.maximum(C_wx2 / w - mu ** 2, 1e-6)
    sigma = np.sqrt(var)
    return A, pi, mu, sigma

def _kl_row(p, q):
    mask = p > 1e-15
    return float(np.sum(p[mask] * (np.log(p[mask]) - np.log(np.maximum(q[mask], 1e-15)))))

def m_step_kl(C_A_num, C_pi, C_w, C_wx, C_wx2, A_old, epsilon):
    d = C_A_num.shape[0]
    A_uncon = np.maximum(C_A_num / np.maximum(C_A_num.sum(1, keepdims=True), EPS_FLOAT), EPS_FLOAT)
    A_uncon /= A_uncon.sum(1, keepdims=True)
    A_new = np.empty((d, d))
    for i in range(d):
        a_old = np.maximum(A_old[i], 1e-10); a_old /= a_old.sum()
        if _kl_row(A_uncon[i], a_old) <= epsilon:
            A_new[i] = A_uncon[i]; continue
        c = np.maximum(C_A_num[i], 1e-10); c /= c.sum()
        def make_row(lam):
            row = a_old ** (1.0 - 1.0/lam) * c ** (1.0/lam)
            row = np.maximum(row, 1e-10); row /= row.sum(); return row
        lo, hi = 1.0, 1e6
        for _ in range(BIN_SEARCH_MAXITER):
            mid = math.sqrt(lo * hi)
            kl = _kl_row(make_row(mid), a_old)
            if abs(kl - epsilon) < BIN_SEARCH_TOL: break
            if kl > epsilon: lo = mid
            else: hi = mid
        A_new[i] = make_row(math.sqrt(lo * hi))
    A_new = np.maximum(A_new, EPS_FLOAT); A_new /= A_new.sum(1, keepdims=True)
    pi = np.maximum(C_pi / max(C_pi.sum(), EPS_FLOAT), EPS_FLOAT); pi /= pi.sum()
    w = np.maximum(C_w[:, None], EPS_FLOAT)
    mu = C_wx / w
    var = np.maximum(C_wx2 / w - mu ** 2, 1e-6)
    sigma = np.sqrt(var)
    return A_new, pi, mu, sigma

print('✓ M-step functions defined')

## Viterbi Decoding & Accuracy

In [ ]:
def viterbi(obs, A, pi, mu, sigma):
    log_B = np.log(np.maximum(emission_prob(obs, mu, sigma), EPS_FLOAT))
    log_A = np.log(np.maximum(A, EPS_FLOAT))
    log_pi = np.log(np.maximum(pi, EPS_FLOAT))
    T, d = log_B.shape
    delta, psi = np.empty((T, d)), np.zeros((T, d), dtype=int)
    delta[0] = log_pi + log_B[0]
    for t in range(1, T):
        sc = delta[t-1, :, None] + log_A
        psi[t] = sc.argmax(0)
        delta[t] = log_B[t] + sc.max(0)
    path = np.empty(T, dtype=int)
    path[-1] = delta[-1].argmax()
    for t in range(T-2, -1, -1):
        path[t] = psi[t+1, path[t+1]]
    return path

def _match_states(pred, true, d):
    conf = np.zeros((d, d), dtype=int)
    for p, t in zip(pred, true): conf[p, t] += 1
    mapping, used = {}, set()
    for _ in range(d):
        best_val, bp, bt = -1, -1, -1
        for pp in range(d):
            if pp in mapping: continue
            for tt in range(d):
                if tt in used: continue
                if conf[pp, tt] > best_val: best_val, bp, bt = conf[pp, tt], pp, tt
        if bp < 0: break
        mapping[bp] = bt; used.add(bt)
    rem = [t for t in range(d) if t not in used]
    for pp in range(d):
        if pp not in mapping: mapping[pp] = rem.pop() if rem else 0
    return np.array([mapping[p] for p in pred], dtype=int)

def viterbi_accuracy(test_obs_list, true_states_list, model):
    total_c = total_s = 0
    for obs, true_s in zip(test_obs_list, true_states_list):
        path = viterbi(obs, model['A'], model['pi'], model['mu'], model['sigma'])
        path_m = _match_states(path, true_s, D)
        total_c += (path_m == true_s).sum(); total_s += len(true_s)
    return total_c / max(total_s, 1)

def compute_ll_batched(obs_array, A, pi, mu, sigma):
    N, T, _ = obs_array.shape
    B = emission_prob(obs_array, mu, sigma)
    alpha = pi[None] * B[:, 0]
    scales_total = 0.0
    c = alpha.sum(1)
    scales_total += float(np.log(np.maximum(c, EPS_FLOAT)).sum())
    alpha /= np.maximum(c[:, None], EPS_FLOAT)
    for t in range(1, T):
        alpha = B[:, t] * (alpha @ A)
        c = alpha.sum(1)
        scales_total += float(np.log(np.maximum(c, EPS_FLOAT)).sum())
        alpha /= np.maximum(c[:, None], EPS_FLOAT)
    return scales_total

print('✓ Viterbi functions defined')

In [ ]:
rng = np.random.default_rng(99)
d, obs_dim, T, N = 3, 2, 8, 4
A = rng.dirichlet(np.ones(d)*2, size=d)
pi = rng.dirichlet(np.ones(d)*2)
mu = rng.standard_normal((d, obs_dim))
sigma = np.ones((d, obs_dim))
obs_arr = rng.standard_normal((N, T, obs_dim))

B = emission_prob(obs_arr, mu, sigma)
assert B.shape == (N, T, d)
C_A, C_pi, C_w, C_wx, C_wx2, ll = batched_e_step(obs_arr, A, pi, mu, sigma)
assert math.isfinite(ll)
A2, pi2, mu2, sigma2 = m_step_standard(C_A, C_pi, C_w, C_wx, C_wx2)
assert np.allclose(A2.sum(1), 1.0, atol=1e-8)
path = viterbi(obs_arr[0], A, pi, mu, sigma)
assert path.shape == (T,)

print('✓ All unit tests passed')

## Run One Config

In [ ]:
def run_config(k_true, T, seed):
    rng = np.random.default_rng(seed * 10000 + k_true * 1000 + T)
    params = make_random_hmm_params(rng, D, OBS_DIM)
    trans_tensor = params['A'].copy() if k_true == 1 else make_kth_order_transition(rng, D, k_true)
    
    obs_all, states_all = generate_sequences(rng=rng, n_seqs=N_TRAIN+N_TEST, T=T, k_true=k_true, params=params, trans_tensor=trans_tensor)
    train_arr = np.stack(obs_all[:N_TRAIN])
    test_arr = np.stack(obs_all[N_TRAIN:])
    test_obs_list = obs_all[N_TRAIN:]
    test_states_list = states_all[N_TRAIN:]
    
    if k_true == 1:
        A_true = params['A']
    else:
        counts = np.ones((D, D))
        for states in states_all:
            for t in range(len(states)-1):
                counts[states[t], states[t+1]] += 1
        A_true = counts / counts.sum(1, keepdims=True)
    
    def run_bw(train_arr, test_arr, epsilon=None, init_from=None):
        A = init_from['A'].copy() if init_from else rng.dirichlet(np.ones(D)*1.5, size=D)
        if not init_from: A = np.maximum(A, EPS_FLOAT); A /= A.sum(1, keepdims=True)
        pi = init_from['pi'].copy() if init_from else rng.dirichlet(np.ones(D)*2)
        mu = init_from['mu'].copy() if init_from else rng.standard_normal((D, OBS_DIM))*2
        sigma = init_from['sigma'].copy() if init_from else np.ones((D, OBS_DIM))*1.5
        
        prev_ll = -np.inf
        for n_iter in range(MAX_ITER):
            try:
                C_A, C_pi, C_w, C_wx, C_wx2, train_ll = batched_e_step(train_arr, A, pi, mu, sigma)
            except: break
            if not math.isfinite(train_ll): break
            
            if epsilon is None: A, pi, mu, sigma = m_step_standard(C_A, C_pi, C_w, C_wx, C_wx2)
            else: A, pi, mu, sigma = m_step_kl(C_A, C_pi, C_w, C_wx, C_wx2, A_old=A, epsilon=epsilon)
            
            if abs(train_ll - prev_ll) < CONV_TOL * (abs(prev_ll)+1): break
            prev_ll = train_ll
        
        try: test_ll = compute_ll_batched(test_arr, A, pi, mu, sigma)
        except: test_ll = -np.inf
        
        return {'A': A, 'pi': pi, 'mu': mu, 'sigma': sigma, 'test_ll': test_ll, 'n_iter': n_iter+1} if math.isfinite(test_ll) else None
    
    bw = run_bw(train_arr, test_arr)
    bw_acc = viterbi_accuracy(test_obs_list, test_states_list, bw) if bw else 0.0
    
    kl_results = {}
    for eps in EPSILONS:
        kl = run_bw(train_arr, test_arr, epsilon=eps, init_from=bw) if bw else None
        acc = viterbi_accuracy(test_obs_list, test_states_list, kl) if kl else 0.0
        kl_results[f'{eps}'] = acc
    
    best_kl_acc = max(kl_results.values()) if kl_results else 0.0
    return {'k_true': k_true, 'T': T, 'seed': seed, 'bw_acc': bw_acc, 'kl_best': best_kl_acc}

print('✓ Config runner defined')

## Execute Experiment

In [ ]:
print('='*70)
print('KL-Constrained BW Demo (Minimal Config)')
print('='*70)
t0 = time.time()

all_results = []
for k in K_TRUES:
    for T in T_VALUES:
        for s in SEEDS:
            r = run_config(k, T, s)
            all_results.append(r)
            print(f"k={k} T={T} seed={s}: BW={r['bw_acc']:.3f} KL={r['kl_best']:.3f}")

elapsed = time.time() - t0
print('='*70)
print(f'Total: {len(all_results)} results in {elapsed:.1f}s')

## Results Summary

In [ ]:
df = pd.DataFrame(all_results)
print('\nResults Table:')
print(df.to_string(index=False))

print('\nSummary:')
for k in K_TRUES:
    sub = df[df['k_true'] == k]
    if len(sub) > 0:
        gain = (sub['kl_best'].mean() - sub['bw_acc'].mean()) / max(sub['bw_acc'].mean(), 1e-6) * 100
        print(f"k={k}: BW={sub['bw_acc'].mean():.4f} KL={sub['kl_best'].mean():.4f} gain={gain:.1f}%")

## Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bw_vals = df['bw_acc'].values
kl_vals = df['kl_best'].values

ax1.scatter(bw_vals, kl_vals, s=80, alpha=0.6)
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('Standard BW'); ax1.set_ylabel('KL-BW')
ax1.set_title('Viterbi Accuracy'); ax1.grid(alpha=0.3)

gain = kl_vals - bw_vals
colors = ['g' if x > 0 else 'r' for x in gain]
ax2.bar(range(len(gain)), gain, color=colors, alpha=0.6)
ax2.axhline(0, color='k', linestyle='-', lw=0.5)
ax2.set_ylabel('Improvement'); ax2.set_title('KL Gain')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('demo_results.png', dpi=100)
print('✓ Plot saved')